###Import Libraries

In [119]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import altair as alt
import plotly.graph_objects as go
from vega_datasets import data
import plotly.express as px
from folium.plugins import HeatMap

###Load Datasets

In [120]:
file_path = 'gs://data-vis-open-access-p2/spotify/fan_insights_2026.csv'
fans = pd.read_csv(file_path, encoding='latin-1')

In [121]:
file_path2 = 'gs://data-vis-open-access-p2/spotify/coordinates_supplemental.csv'
cities = pd.read_csv(file_path2, encoding='latin-1')

###Explore Data

In [122]:
fans.head()

,City_ID,City_Name,Country,Monthly_Listeners,Repeat_Listeners,Total_Streams,Discovery_Source,Platform_Type,Avg_Skip_Rate,Save_Rate,Growth_MoM,Venue_Potential
0,SAO_9167,Sao Paulo,Brazil,1159204,421978,16574515,Discover Weekly,Mobile,24.91%,16.13%,5.63%,Stadium
1,MAN_4717,Manila,Philippines,83991,83230,1440057,Discover Weekly,Smart Speaker,19.55%,19.89%,11.74%,Club
2,LIM_3654,Lima,Peru,40653,40653,753300,Discover Weekly,Desktop,15.61%,17.78%,15.65%,Club
3,JOH_2915,Johannesburg,South Africa,707560,284280,4837605,Radio/External,Mobile,32.16%,7.86%,6.95%,Stadium
4,OSA_2898,Osaka,Japan,225304,24692,856141,TikTok Viral,Mobile,53.66%,3.99%,115.56%,Arena


In [123]:
cities.head()

,City_Name,Country,Latitude,Longitude
0,Sao Paulo,Brazil,-23.5505,-46.6333
1,Manila,Philippines,14.5995,120.9842
2,Lima,Peru,-12.0464,-77.0428
3,Johannesburg,South Africa,-26.2041,28.0473
4,Osaka,Japan,34.6937,135.5023


### Clean Data

#### Convert Data Types

In [124]:
fans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   City_ID            1000 non-null   object
 1   City_Name          1000 non-null   object
 2   Country            1000 non-null   object
 3   Monthly_Listeners  1000 non-null   int64 
 4   Repeat_Listeners   1000 non-null   int64 
 5   Total_Streams      1000 non-null   int64 
 6   Discovery_Source   1000 non-null   object
 7   Platform_Type      1000 non-null   object
 8   Avg_Skip_Rate      1000 non-null   object
 9   Save_Rate          1000 non-null   object
 10  Growth_MoM         1000 non-null   object
 11  Venue_Potential    1000 non-null   object
dtypes: int64(3), object(9)
memory usage: 93.9+ KB


The columns that require data type conversion are Avg_Skip_Rate, Save_Rate, and Growth_MoM, as they are currently stored as object (strings) instead of numerical values.

In [125]:
fans['Avg_Skip_Rate'] = fans['Avg_Skip_Rate'].str.replace('%', '').astype(float)
fans['Save_Rate'] = fans['Save_Rate'].str.replace('%', '').astype(float)
fans['Growth_MoM'] = fans['Growth_MoM'].str.replace('%', '').astype(float)

We removed the "%" character from the percentage columns in text strings, converting them into numerical floats. Pandas cannot perform math on symbols, so we have to convert these data types from strings to floats in order to calculate aggregate functions and feature engineer in the future using these columns.

In [126]:
fans.head()

,City_ID,City_Name,Country,Monthly_Listeners,Repeat_Listeners,Total_Streams,Discovery_Source,Platform_Type,Avg_Skip_Rate,Save_Rate,Growth_MoM,Venue_Potential
0,SAO_9167,Sao Paulo,Brazil,1159204,421978,16574515,Discover Weekly,Mobile,24.91,16.13,5.63,Stadium
1,MAN_4717,Manila,Philippines,83991,83230,1440057,Discover Weekly,Smart Speaker,19.55,19.89,11.74,Club
2,LIM_3654,Lima,Peru,40653,40653,753300,Discover Weekly,Desktop,15.61,17.78,15.65,Club
3,JOH_2915,Johannesburg,South Africa,707560,284280,4837605,Radio/External,Mobile,32.16,7.86,6.95,Stadium
4,OSA_2898,Osaka,Japan,225304,24692,856141,TikTok Viral,Mobile,53.66,3.99,115.56,Arena


In [127]:
cities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   City_Name  30 non-null     object 
 1   Country    30 non-null     object 
 2   Latitude   30 non-null     float64
 3   Longitude  30 non-null     float64
dtypes: float64(2), object(2)
memory usage: 1.1+ KB


The cities dataset is already in a clean, usable format because the geographic coordinates are correctly stored as float64 for mapping, while the city and country names are properly categorized as object (string) types

#### Check for Duplicate Rows

In [128]:
# identifies rows where every single column value is exact match with previous row and stores those repeated records
duplicates = fans[fans.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [City_ID, City_Name, Country, Monthly_Listeners, Repeat_Listeners, Total_Streams, Discovery_Source, Platform_Type, Avg_Skip_Rate, Save_Rate, Growth_MoM, Venue_Potential]
Index: []


Since the output is an empty list, it means the fans dataset contains zero exact row duplicates, confirming that every record is technically unique across all its combined fields.

In [129]:
# checks for any duplicate values within City_ID
is_repeated = fans['City_ID'].duplicated().any()
print(f"Are there repeated City IDs? {is_repeated}")

Are there repeated City IDs? False


Since the result is False, it means every single record in the fans dataset has a unique identifier, confirming that no row was accidentally entered twice under the same ID. For example, the 15 Los Angeles rows are actually distinct data points (like specific sub-districts) rather than technical errors or duplicate logs.

In [130]:
# identifies rows where every single column value is exact match with previous row and stores those repeated records
duplicates2 = cities[cities.duplicated()]
print(duplicates2)

Empty DataFrame
Columns: [City_Name, Country, Latitude, Longitude]
Index: []


In [131]:
# checks for any duplicate values within City_ID
is_repeated2 = cities['City_Name'].duplicated().any()
print(f"Are there repeated City Names? {is_repeated2}")

Are there repeated City Names? False


The cities dataset contains zero exact row duplicates, confirming that every record is technically unique across all its combined fields. Also, we have confirmed that there are no repeated city names in the datset.

#### Check for Erroneous Data Points


In [132]:
fans.describe()

,Monthly_Listeners,Repeat_Listeners,Total_Streams,Avg_Skip_Rate,Save_Rate,Growth_MoM
count,1.000000e+03,1.000000e+03,1.000000e+03,1000.000000,1000.000000,1000.000000
mean,6.009625e+05,2.320847e+05,4.823776e+06,29.551020,10.507090,36.170830
std,4.993211e+05,2.605540e+05,4.862514e+06,16.722802,8.888018,48.237367
min,7.954000e+03,5.421000e+03,1.098180e+05,0.100000,0.100000,-34.600000
25%,9.913925e+04,3.654700e+04,1.015954e+06,15.505000,3.985000,1.980000
50%,5.653165e+05,7.121850e+04,2.294861e+06,28.065000,7.575000,12.765000
75%,9.203868e+05,4.290035e+05,8.290792e+06,43.872500,16.157500,68.735000
max,2.903227e+06,1.237165e+06,2.160016e+07,71.470000,34.820000,160.340000


Based on the output, the dataset doesn't appear to have any erroneous values as the all minimum values for listeners and streams are positive and skip/save rates strictly fall within the realistic range of 0% to 100%, the numerical data seems to make sense. The maximum Growth_MoM is 160.34%, which is a reasonable amount for a viral artist and should be kept rather than eliminated as an error.

In [133]:
# check where repeat listeners exceed monthly listeners
logical_errors = fans[fans['Repeat_Listeners'] > fans['Monthly_Listeners']]
print(f"Number of rows where repeats > monthly: {len(logical_errors)}")

Number of rows where repeats > monthly: 0


The number of records where the Repeat_Listeners count exceeds the Monthly_Listeners total is computed and shown by this code. Since it is impossible to have more returning fans than the total number of unique listeners for that month, a result of zero indicates that the data is logically acceptable.

In [134]:
cities.describe(include = "all")

,City_Name,Country,Latitude,Longitude
count,30,30,30.000000,30.000000
unique,30,26,NaN,NaN
top,Sao Paulo,USA,NaN,NaN
freq,1,3,NaN,NaN
mean,NaN,NaN,19.409583,14.660757
std,NaN,NaN,31.491539,83.224071
min,NaN,NaN,-37.813600,-118.243700
25%,NaN,NaN,-3.478850,-67.597375
50%,NaN,NaN,34.372950,7.047050
75%,NaN,NaN,43.209425,93.595775


The latitude and longitude values fall strictly within the valid global ranges of -90 to 90 and -180 to 180 respectively, ensuring all coordinates are physically possible. Thus, the summary confirms that there are no erroneous data points.

### Data Integration

In [135]:
cities_fans_merged= fans.merge(
    cities,
    left_on='City_Name',
    right_on='City_Name',
    how='left'
)
cities_fans_merged = cities_fans_merged.drop(columns=['Country_y'])

cities_fans_merged = cities_fans_merged.rename(columns={'Country_x': 'Country'})

In [136]:
cities_fans_merged.head(5)

,City_ID,City_Name,Country,Monthly_Listeners,Repeat_Listeners,Total_Streams,Discovery_Source,Platform_Type,Avg_Skip_Rate,Save_Rate,Growth_MoM,Venue_Potential,Latitude,Longitude
0,SAO_9167,Sao Paulo,Brazil,1159204,421978,16574515,Discover Weekly,Mobile,24.91,16.13,5.63,Stadium,-23.5505,-46.6333
1,MAN_4717,Manila,Philippines,83991,83230,1440057,Discover Weekly,Smart Speaker,19.55,19.89,11.74,Club,14.5995,120.9842
2,LIM_3654,Lima,Peru,40653,40653,753300,Discover Weekly,Desktop,15.61,17.78,15.65,Club,-12.0464,-77.0428
3,JOH_2915,Johannesburg,South Africa,707560,284280,4837605,Radio/External,Mobile,32.16,7.86,6.95,Stadium,-26.2041,28.0473
4,OSA_2898,Osaka,Japan,225304,24692,856141,TikTok Viral,Mobile,53.66,3.99,115.56,Arena,34.6937,135.5023


In [137]:
cities_fans_merged['Total_Streams_Str'] = cities_fans_merged['Total_Streams'].astype(str)

# 2. Calculate the frequency of each unique stream count
# This will show you if any two cities have the exact same number of streams
stream_frequencies = cities_fans_merged['Total_Streams_Str'].value_counts()

# 3. Display the result
print("Frequency of Total Stream Values:")
print(stream_frequencies)

Frequency of Total Stream Values:
Total_Streams_Str
8367219     1
16574515    1
1440057     1
753300      1
4837605     1
           ..
268320      1
315831      1
1681569     1
687889      1
13725267    1
Name: count, Length: 1000, dtype: int64


###Feature Engineering

In [138]:
city_profiles = cities_fans_merged.groupby(['City_Name', 'Country', 'Latitude', 'Longitude']).agg({
    'Monthly_Listeners': 'mean',
    'Repeat_Listeners': 'mean',
    'Total_Streams': 'sum',
    'Avg_Skip_Rate': 'mean',
    'Save_Rate': 'mean',
    'Growth_MoM': 'mean'
}).reset_index()

We aggregated first to collapse the raw records into 30 unique city profiles, ensuring that the subsequent feature engineering is performed on stable, city-level totals rather than individual rows. By using averages for rates and sum for annual streams, we correctly normalize the different time scales in your data to create a mathematically accurate baseline for comparing engagement across different markets.

In [139]:
city_profiles.head()

,City_Name,Country,Latitude,Longitude,Monthly_Listeners,Repeat_Listeners,Total_Streams,Avg_Skip_Rate,Save_Rate,Growth_MoM
0,Amsterdam,Netherlands,52.3676,4.9041,640446.071429,227869.928571,140161021,31.031429,9.709286,38.945000
1,Bangkok,Thailand,13.7563,100.5018,431947.827586,186400.275862,111355955,28.579655,12.890000,35.313448
2,Berlin,Germany,52.5200,13.4050,673305.944444,262602.305556,212959489,31.571111,9.686667,40.471111
3,Bogota,Colombia,4.7110,-74.0721,610716.300000,196846.366667,136737885,29.936667,10.209000,35.512000
4,Buenos Aires,Argentina,-34.6037,-58.3816,597043.444444,258020.333333,141670322,27.804815,11.021852,30.954444


After aggregation, each row displays information of a city and its Spotify statistics.

In [140]:
city_profiles['Monthly_Streams'] = city_profiles['Total_Streams'] / 12

This step normalizes the annual Total_Streams figure into a monthly average, allowing for proper comparison with Monthly_Listeners and other monthly engagement metrics.

In [141]:
city_profiles['Repeat_Fan_Ratio'] = (city_profiles['Repeat_Listeners'] / city_profiles['Monthly_Listeners']) * 100

This calculation creates a retention metric that determines the percentage of the total audience who are returning fans, directly quantifying the "die-hard" loyalty our client is looking to discover.

In [142]:
city_profiles['Engagement_Ratio'] = city_profiles['Repeat_Listeners'] / city_profiles['Total_Streams']

This ratio identifies conversion efficiency by calculating how many repeat, loyal fans are generated for every stream, in order to compare Engagement vs. Volume.
Because massive cities will always appear to have higher raw engagement/volume values (more people), we should divide Engagement by Volume, allowing us to mathematically prove that a smaller "Cult Hub" is actually more efficient at turning casual listeners into die-hard fans than a major city.

In [143]:
city_profiles['Loyalty_Score'] = (city_profiles['Repeat_Fan_Ratio'] * city_profiles['Save_Rate']) / (city_profiles['Avg_Skip_Rate'])

This formula creates a weighted score that rewards positive engagement (saves and repeats) while heavily penalizing disinterest (skips), effectively ranking cities by their actual devotion. Mathematically, this makes sense because it treats Avg_Skip_Rate as an inverse multiplier, ensuring that even a high-volume city cannot achieve a top score if its audience is prone to skipping tracks.

In [144]:
city_profiles.head()

,City_Name,Country,Latitude,Longitude,Monthly_Listeners,Repeat_Listeners,Total_Streams,Avg_Skip_Rate,Save_Rate,Growth_MoM,Monthly_Streams,Repeat_Fan_Ratio,Engagement_Ratio,Loyalty_Score
0,Amsterdam,Netherlands,52.3676,4.9041,640446.071429,227869.928571,140161021,31.031429,9.709286,38.945000,1.168009e+07,35.579878,0.001626,11.132430
1,Bangkok,Thailand,13.7563,100.5018,431947.827586,186400.275862,111355955,28.579655,12.890000,35.313448,9.279663e+06,43.153424,0.001674,19.463063
2,Berlin,Germany,52.5200,13.4050,673305.944444,262602.305556,212959489,31.571111,9.686667,40.471111,1.774662e+07,39.001929,0.001233,11.966594
3,Bogota,Colombia,4.7110,-74.0721,610716.300000,196846.366667,136737885,29.936667,10.209000,35.512000,1.139482e+07,32.232047,0.001440,10.991771
4,Buenos Aires,Argentina,-34.6037,-58.3816,597043.444444,258020.333333,141670322,27.804815,11.021852,30.954444,1.180586e+07,43.216341,0.001821,17.130994


### Data Visualization

#### The Listener Journey

In [145]:
# aggregate data by discovery source
source_totals = cities_fans_merged.groupby('Discovery_Source')['Monthly_Listeners'].sum().reset_index()
source_totals = source_totals.sort_values(by='Monthly_Listeners', ascending=False).reset_index(drop=True)

# sources for total audience
unique_sources = source_totals['Discovery_Source'].tolist()
all_nodes = unique_sources + ["Total Artist Audience"]

# flow
sources = []
targets = []
values = []

for i, row in source_totals.iterrows():
    sources.append(i)
    targets.append(len(all_nodes) - 1)
    values.append(row['Monthly_Listeners'])

# sankey visualization
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 30,
      thickness = 30,
      line = dict(color = "black", width = 0.5),
      label = all_nodes,
      color = ['#440154', '#3b528b', '#21918c', '#5ec962', '#fde725', '#333333']
    ),
    link = dict(
      source = sources,
      target = targets,
      value =  values,
      color = 'rgba(200, 200, 200, 0.4)'
  ))])

fig.update_layout(
    title_text="<b>Top Discovery Drivers:</b> External Reach and Editorial Playlists Dominate the Listener Funnel",
    font_size=14,
    width=900,
    height=500
)

fig.show()

We chose a Sankey Diagram to visualize the listener journey because its flow structure intuitively maps the relative weight of diverse discovery sources into the artist's total audience. By ranking these sources from largest to smallest, the chart creates a clear visual hierarchy that immediately identifies Radio/External, Editorial Playlists, and Tiktok as the primary volume drivers for listeners. This funnel approach allows us to see exactly how fragmented discovery journeys converge into a singular, unified "Total Artist Audience," which is critical for identifying where to allocate marketing resources if requested.

The Sankey Visualization demonstrated that the artist's discovery is heavily driven by Radio/External and Editorial Playlists, which together account for the vast majority of the Total Artist Audience. While TikTok Viral demonstrate a growing cultural footprint, the "long tail" of algorithm-based sources like Discover Weekly and Daily Mix shows that many fans are still in the early, passive stages of the listener journey. This identifies a clear strategic opportunity to convert the massive volume from external broadcasting into the more loyal, search-driven engagement.

#### Geospatial Density Mapping

In [146]:
# world map
world_topo = data.world_110m.url
countries = alt.topo_feature(world_topo, 'countries')

# base layer (world map)
base_map = alt.Chart(countries).mark_geoshape(
    fill='#e6e6e6',
    stroke='#ffffff',
    strokeWidth=0.5
).project('equirectangular').properties(
    width=900,
    height=500
)


data_bubbles = alt.Chart(cities_fans_merged).mark_circle(
    opacity=0.75,
    stroke='white',
    strokeWidth=0.5
).encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    size=alt.Size('Monthly_Listeners:Q',
                  title='Metro Area Listeners',
                  scale=alt.Scale(range=[10, 800])),
    color=alt.Color('Venue_Potential:N',
                    title='Venue Type',
                    # sorted
                    sort=['Stadium', 'Arena', 'Theater', 'Club'],
                    scale=alt.Scale(scheme='tableau10')),
    order=alt.Order('Monthly_Listeners:Q', sort='descending'),
    tooltip=['City_Name', 'Venue_Potential', 'Monthly_Listeners']
)

# layer both
final_layered_map = (base_map + data_bubbles).properties(
    title={
        "text": "Global Venue Alignment: High Listener Density Justifies Stadium-Scale Touring in Key Hubs "
    }
).configure_view(strokeWidth=0)

final_layered_map.display()

alt.LayerChart(...)

We chose the Layered Bubble approach to represent market depth by nesting multiple venue tiers within a single geographic coordinate, effectively justifying high-capacity bookings like Stadiums through aggregated listener density. This method prevents visual clutter from overlapping data points while highlighting the diverse venue types available in each major city. By sorting the bubbles from largest to smallest, we maintain a clear visual hierarchy that instantly identifies the primary powerhouse markets for the world tour.

This visualization justifies our booking strategy by demonstrating that global listener density directly aligns with the existing venue infrastructure in each city. For high-volume hubs like São Paulo and Jakarta, the massive outer-ring diameter proves we have the 2M+ monthly listener base necessary to sell out a Stadium (blue ring) without overextending. In secondary markets, the presence of prominent Arena (orange) and Theater (red) layers confirms we can sustain high-capacity, multi-night residencies where stadium infrastructure is either unavailable or unnecessary. By visualizing these nested layers, we mitigate tour risk, ensuring we only book top-tier venues in cities where the combined metropolitan data confirms a high level of demand.

#### Engagement vs. Volume

In [147]:
# scatter plot
scatter = alt.Chart(city_profiles).mark_circle(
    size=450,
    opacity=0.9,
    stroke='white',
    strokeWidth=1
).encode(
    x=alt.X('Total_Streams:Q',
        title='Annual Consumption (Total Streams)',
        scale=alt.Scale(domain=[70000000, 260000000], clamp=True),
        axis=alt.Axis(format='~s')),
    y=alt.Y('Save_Rate:Q',
            title='Engagement (Avg. Save Rate %)',
            scale=alt.Scale(zero=False)),
    color=alt.Color('Repeat_Fan_Ratio:Q',
                    title='Repeat Rate (%)',
                    scale=alt.Scale(scheme='redyellowgreen')),
    tooltip=[
        alt.Tooltip('City_Name:N', title='City'),
        alt.Tooltip('Total_Streams:Q', title='Total Streams', format=',d'),
        alt.Tooltip('Save_Rate:Q', title='Save Rate (%)', format='.1f'),
        alt.Tooltip('Repeat_Fan_Ratio:Q', title='Repeat Fan Ratio (%)', format='.1f')
    ]
)

final_matrix = (scatter).properties(
    width=850,
    height=550,
    title="Weak, Inverse Relationship Between Market Scale and Fan Engagement Exists"
).configure_view(
    strokeWidth=0
)

final_matrix.display()

alt.Chart(...)

We chose a scatterplot to visualize the relationship between market scale (total streams) and engagement in terms of average save rate. By employing a Red-Yellow-Green divergent color scale, we created an intuitive system where high-loyalty cities (cities with a high repeat song ratio) are immediately identifiable through vibrant green colors. Our design ensures that every visual element—from the uniform circle sizing to the high-contrast color mapping—is dedicated to surfacing the most profitable hubs for our global tour.

This graph proves that there is a weak negative correlation between cumulative consumption volume (total streams) and fan loyalty (engagement). We chose to visualize Total Annual Streams on the x-axis to expose the "Dilution Effect" that often occurs when an artist reaches high-population, mainstream cities. The cities with the high repeat ratio and save rate (Johannesburg, Paris, Bangkok) indicate further investigation, as these cities are where loyal fans reside despite their low total streams. Ultimately, this visualization proves that Volume is not a proxy for Loyalty, allowing us to prioritize tour stops based on loyalty rather than volume. Additionally, our analysis identified that  repeat fan ratios are distributed across the entire negative-correlation trend, indicating that high-loyalty hubs can emerge independently of a city's total stream volume. By visualizing the lack of a cluster, we identified cities that over-index on loyalty despite lower total reach, providing a roadmap for tour prioritization.

#### Other Visualizations

In [148]:
# sort by loyalty score and show others stats
city_profiles_sorted = city_profiles.sort_values(by='Loyalty_Score', ascending=False)

city_profiles_sorted = city_profiles_sorted.reset_index(drop=True)

print(city_profiles_sorted[['City_Name', 'Repeat_Fan_Ratio', 'Loyalty_Score','Avg_Skip_Rate']].head(10))

       City_Name  Repeat_Fan_Ratio  Loyalty_Score  Avg_Skip_Rate
0   Johannesburg         41.916019      21.901993      24.430385
1        Bangkok         43.153424      19.463063      28.579655
2          Osaka         39.568128      19.342500      24.723939
3          Paris         46.297981      18.456339      28.622143
4   Buenos Aires         43.216341      17.130994      27.804815
5          Lagos         36.982996      16.689409      28.384286
6        Chicago         38.315919      16.503632      28.071429
7         Sydney         38.639739      16.047905      26.775143
8         Mumbai         37.175944      15.980601      26.722821
9  New York City         44.617938      15.317607      29.534000


Based on this data, the presence of cities like Johannesburg and Osaka with high Repeat Fan Ratios and remarkably low Avg Skip Rates ($<25\%$) directly confirms the existence of Archetype C (Cult Classics) in cities often overlooked for their raw volume. Conversely, New York City and Paris exhibit higher audience friction ($>28\%$ skip rate), illustrating the audience dilution that occurs as a fan base scales. To prove this strategically, we should prioritize an Efficiency Matrix that reverses the Skip Rate axis to elevate these low-friction strongholds and a Loyalty Ranker that ignores total population to reveal where the artist's core truly resides.

In [149]:
fig = px.scatter(city_profiles,
                 x="Repeat_Fan_Ratio",
                 y="Avg_Skip_Rate",
                 color="Loyalty_Score",
                 text="City_Name",
                 size="Loyalty_Score",
                 hover_data={
                     "Repeat_Fan_Ratio": ":.1f",
                     "Avg_Skip_Rate": ":.1f",
                     "Loyalty_Score": ":.1f",
                     "City_Name": False
                 },
                 color_continuous_scale=px.colors.diverging.RdYlGn,
                 title="Johannesburg, Osaka, and Paris Show High Loyalty and Less Skips")

fig.update_layout(
    template="plotly_white",
    xaxis_title="Repeat Fan Ratio (%)",
    yaxis_title="Avg Skip Rate (%)"
)

fig.show()

We implemented a normalized scatterplot to visualize the inverse relationship between average skip rate and repeat fan ratio, effectively mapping the efficiency of the artist's global audience. By utilizing a divergent color gradient for the loyalty score, we created a high-fidelity visual hierarchy that identifies markets achieving the "golden ratio"—where low friction (low skips) intersects with high retention (repeat fans). The inclusion of interactive yooltips ensures that executives can drill down into specific city-level performance.

The scatterplot reveals a somewhat weak negative correlation between low skip rates and high repeat fan ratios. The scatterplot displays different clusters of loyalty levels. The high-intensity quadrant identifies cities where the audience is not just discovering the music but actively committing to the artist's brand, such as Johannesburg, Osaka, Paris, etc. Conversely, markets with high skip rates and low retention indicate passive reach cities, where broad visibility has yet to translate into stable, long-term consumer loyalty. However, for the tour's sake, the cities with the high loyalty score, low average skip rate, and high repeat ratio are cities we should focus on.

In [150]:
m = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB dark_matter")

title_html = '''
             <h3 align="center" style="font-size:20px; color:white; background-color: rgba(0,0,0,0.6);
             padding:10px; border-radius:5px; position: fixed; top: 10px; left: 50%;
             transform: translateX(-50%); z-index:9999; font-family: 'Arial', sans-serif;">
             <b>Mapping Regional Audience Retention Hubs By Avg. Skip Rate</b>
             </h3>
             '''
m.get_root().html.add_child(folium.Element(title_html))


min_skip = city_profiles['Avg_Skip_Rate'].min()
max_skip = city_profiles['Avg_Skip_Rate'].max()

heat_data = [
    [
        row['Latitude'],
        row['Longitude'],
        (row['Avg_Skip_Rate'] - min_skip) / (max_skip - min_skip)
    ]
    for idx, row in city_profiles.iterrows()
]

# heatmap
HeatMap(heat_data,
        radius=30,
        blur=20,
        gradient={0.0: 'blue', 0.3: 'cyan', 0.5: 'lime', 0.7: 'orange', 1.0: 'red'},
        min_opacity=0.4).add_to(m)

# marker
for _, row in city_profiles.iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4,
        color='white',
        weight=1,
        fill=True,
        fill_opacity=0.2,
        popup=f"<b>{row['City_Name']}</b><br>Avg. Skip Rate: {row['Avg_Skip_Rate']:.1f}%"
    ).add_to(m)

m

We implemented a global friction heatmap to visualize the geographic distribution of audience engagement, utilizing a warm-to-cool divergent scale to distinguish between high-friction and high-retention regions. By applying min-max normalization to the skip-rate data, we maximized visual contrast within the narrow $24\%$-$35\%$ performance range, ensuring that even subtle behavioral variances are geographically identifiable. The use of a dark matter base map reduces visual noise, allowing the cool zones (blue)—representing markets with elite listener focus—to stand out against warm zones (red) that require strategic intervention. This spatial analysis provides an immediate diagnostic tool for identifying the most receptive global territories for future market expansion.

The global heatmap identifies Johannesburg, Osaka, Sydney, and Mumbai as elite cool zones, characterized by the lowest skip rates in the entire dataset. These regions represent the highest-quality engagement, where listeners are most likely to lean in and finish a track rather than skipping. In contrast, warm zones like North America and parts of Europe show higher audience friction, suggesting that while those markets have high volume, they experience  listener behavior that consists of higher skip rates.

In [151]:
top_5 = city_profiles.nlargest(5, 'Repeat_Fan_Ratio')
bottom_5 = city_profiles.nsmallest(5, 'Repeat_Fan_Ratio')
extreme_cities = pd.concat([top_5, bottom_5])

# gradient bar chart
contrast_chart = alt.Chart(extreme_cities).mark_bar(
    cornerRadiusEnd=4,
    height=20
).encode(
    x=alt.X('Repeat_Fan_Ratio:Q',
            title='Repeat Fan Ratio (%)',
            scale=alt.Scale(domain=[0, 55]),
            axis=alt.Axis(format='.1f', grid=True, tickCount=6)),
    y=alt.Y('City_Name:N',
            sort='-x',
            title=None),
    color=alt.Color('Repeat_Fan_Ratio:Q',
                    title='Repeat_Fan_Ratio',
                    scale=alt.Scale(scheme='redyellowgreen'),
                    legend=alt.Legend(title="Repeat_Fan_Ratio")),
    tooltip=[
        alt.Tooltip('City_Name:N'),
        alt.Tooltip('Repeat_Fan_Ratio:Q', format='.2f', title='Repeat Ratio %'),
        alt.Tooltip('Avg_Skip_Rate:Q', format='.2f', title='Skip Rate %')
    ]
).properties(
    width=600,
    height=400,
    title="Stockholm and Paris Outperform Global Retention Benchmarks"
)

(contrast_chart).configure_view(strokeWidth=0).display()

alt.Chart(...)

We utilized a barchart with a divergent color gradient to provide an analytical comparison of the performance gap between top-tier and lower-tier global markets. By applying a min-max normalization to the color scale, we created a clear visual signal that distinguishes between cities falling below or exceeding the global mean benchmark. This layout ensures that the visual magnitude of the bars accurately reflects the total variance in fan loyalty.

The visualization reveals a significant retention disparity, showing that top-performing markets achieve much higher repeat listener rate of the lowest-performing cities. Markets like Stockholm, Paris, and New York City significantly exceed the global average of 39%, indicating a much more stable and loyal fan base than the general audience. This clear gap highlights the difference between high-engagement strongholds and markets that currently rely on one-time listeners, suggesting a need for distinct marketing strategies for each tier.